In [24]:
import json
import random
from collections import defaultdict
import numpy as np
from pathlib import Path
import pandas as pd

In [ ]:
# !pip install transformers -q

In [6]:
from transformers import AutoTokenizer

In [28]:
TRAIN_JSONL = Path("../seed_exports/splits/train_all.jsonl")
VAL_JSONL = Path("../seed_exports/splits/val_all.jsonl")
SEED = 3407
random.seed(SEED)
MAX_PER_TASK_TRAIN = 5000
MAX_LINES_VAL = None

REPORT_MAX_LENS = (2048, 4096)# Ngưỡng báo cáo truncation

In [18]:
MODEL_ID = "unsloth/Qwen2.5-1.5B-Instruct-unsloth-bnb-4bit"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

In [19]:
tokenizer

Qwen2Tokenizer(name_or_path='unsloth/Qwen2.5-1.5B-Instruct-unsloth-bnb-4bit', vocab_size=151643, model_max_length=32768, padding_side='left', truncation_side='right', special_tokens={'eos_token': '<|im_end|>', 'pad_token': '<|vision_pad|>'}, added_tokens_decoder={
	151643: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151644: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151645: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151646: AddedToken("<|object_ref_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151647: AddedToken("<|object_ref_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151648: AddedToken("<|box_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151649: AddedToken("<|bo

In [20]:
def row_task(row: dict) -> str:
    meta = row.get("metadata") or {}
    return str(meta.get("task", "missing_task"))

def messages_to_text(messages: list[dict]) -> str:
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    # Khớp pipeline train: nếu train có append EOS khi thiếu, làm giống ở đây.
    eos = tokenizer.eos_token
    if eos and not text.rstrip().endswith(eos):
        text = text + eos
    return text

def token_length(text: str) -> int:
    return len(tokenizer(text, add_special_tokens=False)["input_ids"])

In [22]:
def iter_jsonl(path: Path, max_lines: int | None = None):
    n = 0
    with path.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            yield json.loads(line)
            n += 1
            if max_lines is not None and n >= max_lines:
                break


def stratified_sample_train(path: Path, max_per_task: int, seed: int) -> list[dict]:
    rng = random.Random(seed)
    buckets: dict[str, list[dict]] = defaultdict(list)

    for row in iter_jsonl(path):
        t = row_task(row)
        if len(buckets[t]) < max_per_task:
            buckets[t].append(row)
        else:
            j = rng.randrange(max_per_task)
            buckets[t][j] = row

    out: list[dict] = []
    for rows in buckets.values():
        out.extend(rows)
    rng.shuffle(out)
    return out


def load_val_rows(path: Path, max_lines: int | None) -> list[dict]:
    return list(iter_jsonl(path, max_lines=max_lines))

In [29]:
# tính độ dài seq
def measure_split(rows: list[dict], split_name: str) -> pd.DataFrame:
    lengths: list[int] = []
    tasks: list[str] = []
    errors = 0

    for row in rows:
        try:
            messages = row["messages"]
            text = messages_to_text(messages)
            lengths.append(token_length(text))
            tasks.append(row_task(row))
        except Exception:
            errors += 1

    df = pd.DataFrame({"task": tasks, "seq_len": lengths})
    df.attrs["split"] = split_name
    df.attrs["errors"] = errors
    df.attrs["n"] = len(df)
    return df


train_rows = stratified_sample_train(TRAIN_JSONL, MAX_PER_TASK_TRAIN, SEED)
val_rows = load_val_rows(VAL_JSONL, MAX_LINES_VAL)

df_train = measure_split(train_rows, "train")
df_val = measure_split(val_rows, "val")

print(f"train rows: {df_train.attrs['n']}, errors: {df_train.attrs['errors']}")
print(f"val rows:   {df_val.attrs['n']}, errors: {df_val.attrs['errors']}")

train rows: 25000, errors: 0
val rows:   2076, errors: 0


In [30]:
def percentile_summary(arr: np.ndarray) -> dict:
    arr = np.asarray(arr, dtype=np.int64)
    if arr.size == 0:
        return {f"p{q}": float("nan") for q in (50, 90, 95, 99)} | {"max": 0}
    out = {f"p{int(q)}": float(np.percentile(arr, q)) for q in (50, 90, 95, 99)}
    out["max"] = int(arr.max())
    return out


def truncation_rates(arr: np.ndarray, thresholds: tuple[int, ...]) -> dict[str, float]:
    arr = np.asarray(arr, dtype=np.int64)
    if arr.size == 0:
        return {str(t): float("nan") for t in thresholds}
    return {str(t): float((arr > t).mean()) for t in thresholds}


def summarize_by_task(df: pd.DataFrame, thresholds: tuple[int, ...]) -> pd.DataFrame:
    rows_out = []
    for task, g in df.groupby("task", sort=True):
        a = g["seq_len"].to_numpy()
        p = percentile_summary(a)
        tr = truncation_rates(a, thresholds)
        rows_out.append({"task": task, "n": len(g), **p, **{f"trunc>{k}": v for k, v in tr.items()}})

    a_all = df["seq_len"].to_numpy()
    p_all = percentile_summary(a_all)
    tr_all = truncation_rates(a_all, thresholds)
    rows_out.append(
        {"task": "__overall__", "n": len(df), **p_all, **{f"trunc>{k}": v for k, v in tr_all.items()}}
    )
    return pd.DataFrame(rows_out)


sum_train = summarize_by_task(df_train, REPORT_MAX_LENS)
sum_val = summarize_by_task(df_val, REPORT_MAX_LENS)

pd.set_option("display.max_rows", 200)
display(sum_train)
display(sum_val)

,task,n,p50,p90,p95,p99,max,trunc>2048,trunc>4096
0,cloze_lm_retention,5000,151.0,193.0,207.00,234.00,991,0.00000,0.0
1,comprehension_short_answer,5000,307.0,451.0,508.00,627.01,2421,0.00020,0.0
2,exams_mcq,5000,108.0,156.0,179.00,281.02,515,0.00000,0.0
3,instruction_retention,5000,166.0,441.1,528.00,674.02,1375,0.00000,0.0
4,wiki_mcq,5000,112.0,196.1,258.05,513.11,1095,0.00000,0.0
5,__overall__,25000,146.0,356.0,433.05,591.01,2421,0.00004,0.0


,task,n,p50,p90,p95,p99,max,trunc>2048,trunc>4096
0,comprehension_short_answer,1083,307.0,444.0,509.00,612.54,1008,0.0,0.0
1,exams_mcq,597,109.0,152.4,175.20,229.44,496,0.0,0.0
2,wiki_mcq,396,110.0,219.0,290.75,630.35,842,0.0,0.0
3,__overall__,2076,237.0,390.0,455.00,596.75,1008,0.0,0.0


In [31]:
row_all_train = sum_train.loc[sum_train["task"] == "__overall__"].iloc[0]
row_all_val = sum_val.loc[sum_val["task"] == "__overall__"].iloc[0]

print("=== Overall ===")
print("train trunc>4096:", row_all_train["trunc>4096"], "trunc>2048:", row_all_train["trunc>2048"])
print("val   trunc>4096:", row_all_val["trunc>4096"], "trunc>2048:", row_all_val["trunc>2048"])

for name, tbl in ("train", sum_train), ("val", sum_val):
    comp = tbl.loc[tbl["task"] == "comprehension_short_answer"]
    if len(comp):
        r = comp.iloc[0]
        print(f"{name} comprehension_short_answer trunc>4096:", r["trunc>4096"], "n=", int(r["n"]))

=== Overall ===
train trunc>4096: 0.0 trunc>2048: 4e-05
val   trunc>4096: 0.0 trunc>2048: 0.0
train comprehension_short_answer trunc>4096: 0.0 n= 5000
val comprehension_short_answer trunc>4096: 0.0 n= 1083


In [32]:
import dataclasses
from datetime import datetime, timezone

report = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "model_id": MODEL_ID,
    "seed": SEED,
    "train_jsonl": str(TRAIN_JSONL),
    "val_jsonl": str(VAL_JSONL),
    "max_per_task_train": MAX_PER_TASK_TRAIN,
    "max_lines_val": MAX_LINES_VAL,
    "train_rows": df_train.attrs["n"],
    "val_rows": df_val.attrs["n"],
    "thresholds": list(REPORT_MAX_LENS),
    "summary_train": sum_train.to_dict(orient="records"),
    "summary_val": sum_val.to_dict(orient="records"),
}

out_path = Path("token_length_report.json")
out_path.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
print("wrote", out_path.resolve())

wrote /home/tontide1/coding/nlp_project/Multi-task-fine-tuning-of-LLMs-VLSP-2023-VLLMs/notebooks/token_length_report.json
